# Agent 1 — Biomarker Agent (Package Edition)

This notebook validates the **packaged** Agent 1 (Biomarker) against the original exploration notebook (`01_Biomarker_Exploration.ipynb`). Everything that lived as inline code in the exploration is now in `immunosense.agents.biomarker` and exercised through three demonstrations:

1. **Synthetic patient generator** — confirms `ImprovedPatientGenerator` produces realistic flare patterns.
2. **Layer 3 baseline tracker** — walks a 30-reading RA trajectory through `BiomarkerBaselineTracker` and reports anomalies.
3. **BaseAgent contract** — runs the full `BiomarkerAgent.process()` pipeline (Layer 1 + Layer 2 + Layer 3) and shows the 7-dim disease probability vector for the Conductor plus the 128-dim contrastive embedding for JEPA.

**Section 4** runs the real Layer 1 + Layer 2 pipeline if artifacts are present (skipped otherwise with instructions to train them).

## Three-layer architecture

* **Layer 1** — NHANES → 7 LightGBM quantile regressors → `CRPBaseline.percentile(age, sex, bmi, crp)` (population-adjusted percentile)
* **Layer 2** — Three cognitive pillars on the Rheumatic dataset:
  * Pillar A (Spatial): PyTorch contrastive encoder + NT-Xent loss → 128-dim unit-sphere embedding → nearest centroid classification
  * Pillar B (Probabilistic): LightGBM multiclass classifier with `class_weight='balanced'`
  * Pillar C (Explanatory): XGBoost multiclass classifier + SHAP TreeExplainer
  * Fusion: `0.30 * softmax(sims × 5.0) + 0.35 * P_B + 0.35 * P_C`
* **Layer 3** — Personal adaptation via `BiomarkerBaselineTracker` (median + IQR, outlier-resistant) + `PatternDetector` (trigger × biomarker correlation across lags 1-3)

## Three-dimensional output

* `output.vector` — **7-dim disease probabilities** (for Conductor consumption, BaseAgent contract)
* `agent.emit_embedding()` — **128-dim contrastive embedding** (for JEPA training, unit sphere)
* `output.data['layer2']['top_drivers']` — **per-prediction SHAP feature attributions** (top-3 features pushing toward the predicted class)

In [ ]:
# === Package imports ===
from pathlib import Path

import numpy as np
import pandas as pd

from immunosense.agents.biomarker import (
    BiomarkerAgent,
    BiomarkerBaselineTracker,
    CRPBaseline,
    ImprovedPatientGenerator,
    Layer2Bundle,
    PatternDetector,
    PersonalAdaptationEngine,
    fuse_predictions,
    generate_synthetic_trajectory,
    get_crp_percentile,
)
from immunosense.agents.biomarker.constants import (
    ALL_INPUT_FEATURES,
    BIOMARKERS_FOR_TRACKING,
    BIOMARKER_TRIGGERS,
    DISEASE_CLASSES,
    LAYER2_EMBEDDING_DIM,
)

print('Agent 1 — Biomarker imports OK')
print(f'  Disease classes ({len(DISEASE_CLASSES)}): {DISEASE_CLASSES}')
print(f'  Tracked biomarkers ({len(BIOMARKERS_FOR_TRACKING)}): {BIOMARKERS_FOR_TRACKING}')
print(f'  Triggers ({len(BIOMARKER_TRIGGERS)}): {BIOMARKER_TRIGGERS}')
print(f'  Layer 2 input features: {len(ALL_INPUT_FEATURES)} (14 values + 14 missing flags)')
print(f'  Contrastive embedding dim: {LAYER2_EMBEDDING_DIM}')
print(f'  BiomarkerAgent.output_dim = {BiomarkerAgent.output_dim} (disease probabilities)')

In [ ]:
# === Section 1: Synthetic patient generator smoke test ===
# Generate a 30-reading RA trajectory with planted flare patterns.
# Verify CRP/ESR spike during flares (multipliers [2.0, 3.5, 2.0]).

print('=' * 70)
print('SECTION 1: Synthetic patient generator')
print('=' * 70)

trajectory, baseline = generate_synthetic_trajectory(
    disease='Rheumatoid Arthritis', n_readings=30, random_seed=42,
)

print(f'\n  Trajectory length: {len(trajectory)} readings')
print(f'  Personal baseline (remission): CRP={baseline["CRP"]:.2f} mg/L')

flare_readings = [r for r in trajectory if r['is_flare']]
normal_readings = [r for r in trajectory if not r['is_flare']]

print(f'\n  Flare readings: {len(flare_readings)} ({len(flare_readings) / len(trajectory) * 100:.1f}%)')
print(f'  Normal readings: {len(normal_readings)}')

if flare_readings and normal_readings:
    avg_flare_crp = np.mean([r['CRP'] for r in flare_readings])
    avg_normal_crp = np.mean([r['CRP'] for r in normal_readings])
    print(f'\n  CRP during flares: {avg_flare_crp:.2f} mg/L')
    print(f'  CRP normally:      {avg_normal_crp:.2f} mg/L')
    print(f'  Flare/normal ratio: {avg_flare_crp / max(avg_normal_crp, 0.01):.2f}x')
    assert avg_flare_crp > avg_normal_crp, 'Flares must elevate CRP'

# Show first few readings
print(f'\n  First 5 readings:')
print(f'  {"#":>3} {"Day":>5} {"CRP":>7} {"ESR":>7} {"Flare":>6} {"Gluten":>7} {"Sleep":>6}')
print('  ' + '-' * 50)
for i, r in enumerate(trajectory[:5]):
    flare = 'YES' if r['is_flare'] else ''
    gluten = 'X' if r.get('gluten_exposure') else ''
    sleep = 'X' if r.get('poor_sleep') else ''
    print(f'  {i:>3} {r["day"]:>5} {r["CRP"]:>7.2f} {r["ESR"]:>7.2f} {flare:>6} {gluten:>7} {sleep:>6}')

print('\n  PASS: Synthetic generator produces realistic flare patterns')

In [ ]:
# === Section 2: Layer 3 — BiomarkerBaselineTracker on RA trajectory ===
# Walk the 30-reading trajectory through the robust baseline tracker.
# Confirm: median baseline ignores flare spikes, anomaly_score flags critical readings.

print('=' * 70)
print('SECTION 2: Layer 3 BiomarkerBaselineTracker')
print('=' * 70)

tracker = BiomarkerBaselineTracker(biomarkers=BIOMARKERS_FOR_TRACKING)

print(f'\n  {"#":>3} {"Day":>5} {"CRP":>7} {"Base":>7} {"Anom":>7} {"Status":>14} {"Trend":>8} {"Flare":>6}')
print('  ' + '-' * 65)

for i, reading in enumerate(trajectory):
    tracker.update(reading)
    ctx = tracker.get_personal_context(reading)
    if not ctx['has_personal_data'] or 'CRP' not in ctx['biomarkers']:
        continue
    crp = ctx['biomarkers']['CRP']
    flare = 'YES' if reading['is_flare'] else ''
    # Only print interesting rows: flares or non-normal interpretation
    if reading['is_flare'] or crp['interpretation'] != 'NORMAL':
        print(f'  {i + 1:>3} {reading["day"]:>5} {reading["CRP"]:>7.2f} '
              f'{crp["median_baseline"]:>7.2f} {crp["anomaly_score"]:>7.2f} '
              f'{crp["interpretation"]:>14} {crp["trend"]:>8} {flare:>6}')

print(f'\n  Final personal_weight: {ctx["personal_weight"]:.2f}')
print(f'  Outliers excluded for CRP: {crp["outliers_excluded"]}')
print(f'\n  PASS: Robust baseline detects flare anomalies without contamination')

In [ ]:
# === Section 3: Pattern detector — trigger → biomarker correlation ===
# Confirm that planted trigger-flare correlations (70% trigger 2 readings before flare)
# are detected by PatternDetector.

print('=' * 70)
print('SECTION 3: Pattern detector')
print('=' * 70)

detector = PatternDetector(
    biomarkers=BIOMARKERS_FOR_TRACKING,
    triggers=BIOMARKER_TRIGGERS,
)
result = detector.analyze(trajectory)

print(f'\n  Readings analyzed: {result["n_readings_analyzed"]}')
print(f'  Patterns detected: {len(result["patterns"])}')

if result['has_patterns']:
    print(f'\n  Top patterns by |correlation|:')
    print(f'  {"Trigger":<20} {"Biomarker":<10} {"Lag":>4} {"Corr":>7} {"Effect%":>9} {"Strength":<10}')
    print('  ' + '-' * 65)
    for p in result['patterns'][:8]:
        print(f'  {p["trigger"]:<20} {p["biomarker"]:<10} {p["lag_readings"]:>4} '
              f'{p["correlation"]:>7.3f} {p["effect_pct"]:>8.1f}% {p["strength"]:<10}')

if result.get('flare_rule'):
    print(f'\n  Flare detection thresholds:')
    for bm, rule in result['flare_rule'].items():
        print(f'    {bm}: normal={rule["normal_mean"]:.1f} → flare={rule["flare_mean"]:.1f} '
              f'(ratio: {rule["ratio"]:.1f}x, threshold: {rule["threshold"]:.1f})')

print('\n  PASS: PatternDetector picks up planted trigger-flare correlations')

In [ ]:
# === Section 4: BaseAgent contract — 7-dim vector + 128-dim embedding ===
# Construct a BiomarkerAgent (no Layer 1/2 models loaded yet). Run it on a few readings.
# Verify the BaseAgent contract: output_dim, vector shape, emit_embedding dim.

print('=' * 70)
print('SECTION 4: BaseAgent contract')
print('=' * 70)

agent = BiomarkerAgent(patient_id='demo_patient_001')

print(f'\n  Agent ID:         {agent.agent_id}')
print(f'  Agent version:    {agent.agent_version}')
print(f'  Output dim:       {agent.output_dim} (disease probabilities)')
print(f'  Poll frequency:   {agent.poll_frequency}')
print(f'  Embedding dim:    {LAYER2_EMBEDDING_DIM} (via emit_embedding)')

print(f'\n  Before any process() call:')
print(f'    get_output_vector() shape: {agent.get_output_vector().shape}  (zeros)')
print(f'    emit_embedding() shape:    {agent.emit_embedding().shape}  (zeros)')

print(f'\n  Layer 3 (in-memory state) is always active.')
print(f'  Layer 1 + Layer 2 require trained artifacts (next section).')

In [ ]:
# === Section 5: Full L1 + L2 + L3 pipeline (only if artifacts present) ===
# This section is gated on the existence of trained artifacts. If you have
# them, this runs the complete agent end-to-end. Otherwise, it prints
# instructions for training and exits cleanly.

print('=' * 70)
print('SECTION 5: Full L1 + L2 + L3 pipeline (gated on artifacts)')
print('=' * 70)

layer1_dir = Path('./artifacts/agent1_layer1')
layer2_dir = Path('./artifacts/agent1_layer2')

have_layer1 = layer1_dir.exists() and any(layer1_dir.glob('crp_quantile_*.pkl'))
have_layer2 = layer2_dir.exists() and (layer2_dir / 'lgb_model.pkl').exists()

print(f'\n  Layer 1 artifacts: {layer1_dir} — {"FOUND" if have_layer1 else "missing"}')
print(f'  Layer 2 artifacts: {layer2_dir} — {"FOUND" if have_layer2 else "missing"}')

if not have_layer2:
    print(f'\n  To train Layer 2, run from the project root:')
    print('    python -m immunosense.agents.biomarker.layer2.train \\')
    print('      --rheumatic-xlsx ./data/rheumatic/Rheumatic_and_Autoimmune_Disease_Dataset.xlsx')
    print(f'\n  To train Layer 1 (population CRP baseline), run:')
    print('    python -m immunosense.agents.biomarker.layer1.train')
    print(f'\n  Or both at once:')
    print('    python -m immunosense.agents.biomarker.train \\')
    print('      --rheumatic-xlsx ./data/rheumatic/Rheumatic_and_Autoimmune_Disease_Dataset.xlsx')
    print(f'\n  (skipped Section 5 — train Layer 2 first)')
else:
    # Load Layer 2 always; Layer 1 optional
    print(f'\n  Loading Layer 2 bundle...')
    agent.load_models(
        layer1_dir=layer1_dir if have_layer1 else None,
        layer2_dir=layer2_dir,
        require_layer1=have_layer1,
        require_layer2=True,
    )
    print(f'    Loaded {len(agent.layer2.class_names)} disease classes')
    print(f'    Encoder embedding dim: {agent.layer2.encoder.embedding_dim}')
    print(f'    Centroids on unit sphere: {len(agent.layer2.centroids)}')

    # Run a single RA-shaped reading through the agent
    ra_reading = {
        'day': 0,
        'Age': 52, 'ESR': 42, 'CRP': 22.0, 'RF': 85.0, 'Anti-CCP': 90.0,
        'C3': 108.0, 'C4': 23.0,
        'Gender_enc': 1, 'HLA-B27_enc': 0, 'ANA_enc': 0,
        'Anti-Ro_enc': 0, 'Anti-La_enc': 0, 'Anti-dsDNA_enc': 0, 'Anti-Sm_enc': 0,
    }
    output = agent.process({
        'demographics': {'age': 52, 'sex': 2, 'bmi': 26},
        'reading': ra_reading,
    })

    print(f'\n  PROCESS OUTPUT:')
    print(f'    Agent ID:    {output.agent_id}')
    print(f'    Vector dim:  {output.vector_dim}')
    print(f'    Vector sum:  {output.vector.sum():.6f} (should be ~1.0)')
    print(f'    Prediction:  {output.data["layer2"]["prediction"]}')
    print(f'    Confidence:  {output.confidence:.3f}')
    print(f'    Pillars agree: {output.data["layer2"]["pillars_agree"]}')

    print(f'\n  Disease probabilities (7-dim vector to Conductor):')
    for i, c in enumerate(agent.layer2.class_names):
        bar = '#' * int(output.vector[i] * 40)
        print(f'    {c:<35s} {output.vector[i] * 100:>5.1f}% {bar}')

    embedding = agent.emit_embedding()
    print(f'\n  Contrastive embedding (128-dim for JEPA):')
    print(f'    Shape:   {embedding.shape}')
    print(f'    L2 norm: {np.linalg.norm(embedding):.6f} (should be ~1.0)')
    print(f'    First 5: {embedding[:5].round(3).tolist()}')

    print(f'\n  Top SHAP drivers (per-prediction explainability):')
    for d in output.data['layer2']['top_drivers']:
        print(f'    {d["feature_name"]:<25s} = {d["feature_value"]:>8.2f}  '
              f'SHAP {d["shap_value"]:+.4f}  ({d["direction"]})')

    if have_layer1 and 'CRP' in output.data['layer1']:
        l1 = output.data['layer1']['CRP']
        print(f'\n  Layer 1 population context:')
        print(f'    CRP {l1["value"]:.1f} mg/L → {l1["population_percentile"] * 100:.0f}th '
              f'percentile ({l1["interpretation"]})')

    print(f'\n  PASS: Full agent pipeline runs end-to-end')

## Migration complete — Agent 1 (Biomarker)

Exploration notebook → packaged agent with:

- **Layer 1** (Population baseline): `CRPBaseline` with 7 LightGBM quantile regressors trained on NHANES → demographic-aware percentile lookup
- **Layer 2** (Disease intelligence): 6 atomically-loaded artifacts in `Layer2Bundle`:
  - `lgb_model.pkl` (Pillar B: LightGBM, `class_weight='balanced'`)
  - `xgb_model.json` (Pillar C: XGBoost, native JSON for cross-version compat)
  - `encoder.pt` (Pillar A: PyTorch DiseaseEncoder weights, 128-dim unit sphere)
  - `centroids.npz` (per-class normalized centroids)
  - `scaler.pkl` (StandardScaler for encoder input)
  - `label_encoder.pkl` (sklearn LabelEncoder)
- **Layer 3** (Personal adaptation): `BiomarkerBaselineTracker` (median + IQR, outlier-resistant) + `PatternDetector` (trigger × biomarker correlation lags 1-3) → `PersonalAdaptationEngine` orchestrator

**Three-dimensional output preserved:**
- 7-dim disease probabilities → Conductor consumption (`output.vector`)
- 128-dim L2-normalized contrastive embedding → JEPA training (`agent.emit_embedding()`)
- Per-prediction SHAP feature attributions → explainability (`output.data['layer2']['top_drivers']`)

**Training (offline, ~10 minutes on CPU):**
```bash
# Both layers at once
python -m immunosense.agents.biomarker.train \
    --rheumatic-xlsx ./data/rheumatic/Rheumatic_and_Autoimmune_Disease_Dataset.xlsx

# Or each layer separately
python -m immunosense.agents.biomarker.layer1.train
python -m immunosense.agents.biomarker.layer2.train --rheumatic-xlsx ...
```

**Tests:** 127 passing in ~30 seconds (`pytest tests/agents/biomarker/`).